# Graphs — User Guide

`StarLayerGraph` extends rdflib's `Graph` to support RDF 1.2: triple terms, reification, and direction-tagged literals.

A related guide goes into more depth regarding datasets: **[2.a Working with datasets](02a-graphs-datasets.ipynb)** (`StarLayerDataset`, multiple named graphs in one data store).


## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed. 

Run cells from top to bottom — later sections may reuse variables from earlier sections.

In [1]:
from starlayer import StarLayerGraph, Namespace, TripleTerm, DirLangString, Literal

EX = Namespace("http://example.org/")
RDF = Namespace("http://www.w3.org/1999/02/22-rdf-syntax-ns#")

## 1. Graph and literal semantics

StarLayer provides an extension of the rdflib graph model to support RDF 1.2, including:
- Triple terms and statement resources
- Reification via `rdf:reifies` and statement metadata
- Direction-tagged strings such as `"hello"@en--ltr` and `"مرحبا"@ar--rtl`
- Parsing and serialization for RDF 1.2 syntax.  (e.g. Turtle 1.2)

In [2]:
# create the graph, and assign a namespace
g = StarLayerGraph()
g.bind("ex", EX)

# create a triple term
tt = TripleTerm(EX.alice, EX.worksFor, EX.AcmeCorp)

# create a reifier (ex:claim) associated with the triple term and add it to the graph
g.add_reification(EX.claim, tt)
g.add((EX.claim, EX.source, EX.HRSystem))

print("\nResulting graph")
print(g.serialize(format="turtle12"))

# note that creating a reifier for a triple term, does not automatically assert that term into the graph.  
# ex:alice ex:worksFor ex:AcmeCorp is not asserted.  


Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:HRSystem ;
    rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



In [3]:
g = StarLayerGraph()
g.bind("ex", EX)

# add reifiers to the graph
g.add((EX.claim, RDF.reifies, (EX.alice, EX.worksFor, EX.AcmeCorp)))
g.add((EX.other, RDF.reifies, (EX.alice, EX.likes, EX.ProductABC)))

# explicitly adding a triple to the graph.
g.add((EX.alice, EX.worksFor, EX.GlobalTech))

# rdflib triples() now accepts a triple term as the object when selecting triples
# seelects all triples with the triple term as the object
selectTriples = g.triples((None, None, (EX.alice, EX.worksFor, EX.AcmeCorp)))

for s, p, o in selectTriples:
    print(g.qname(s), g.qname(p), o)
for t in g.triple_terms(subject=EX.alice):
    print(t)

# has_triple_term() tests whether the triple term is in the graph.
# (EX.alice, EX.worksFor, EX.GlobalTech) is asserted directly in the graph, but is not the
# object of any triple, so it returns False.
print(g.has_triple_term(EX.alice, EX.worksFor, EX.AcmeCorp))
print(g.has_triple_term(EX.alice, EX.worksFor, EX.GlobalTech))

print("\nResulting graph")
print(g.serialize(format="turtle12"))

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>>
<<( ex:alice ex:worksFor ex:AcmeCorp )>>
<<( ex:alice ex:likes ex:ProductABC )>>
True
False

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:worksFor ex:GlobalTech .

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:other rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



In [4]:
# rdf:reifies is the common approach to making a statement about a triple statement.
# RDF 1.2 allows triple terms in the object position of any triple.

# add a triple to the graph with a triple term as the object
g.add((EX.AuditSystem, EX.reported, (EX.alice, EX.worksFor, EX.AcmeCorp)))

# triples() accepts a triple term as object to select matching triples
selectTriples = g.triples((None, None, (EX.alice, EX.worksFor, EX.AcmeCorp)))

# qname_term() is a starlayer function that adds qname transformation to triple term.
for s, p, o in selectTriples:
    print(g.qname_term(s), g.qname_term(p), g.qname_term(o))

print("\nResulting graph")
print(g.serialize(format="turtle12"))

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>>
ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>>

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:worksFor ex:GlobalTech .

ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:other rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### Working with statements about statements

`reifiers()`, `reifications()`, `reifier_annotations()`, `reified_triples()`, and `remove_reification()` navigate the reifier/triple-term/annotation relationships directly, without using SPARQL queries. 

`remove_reification(reifier, triple_term=None)` can be scoped to one specific reifier↔triple link, leaving any other triple(s) the same reifier reifies — and all its annotations — untouched; omit `triple_term` for the original all-or-nothing behavior.

In [5]:
# continues using g from the previous cell

# adds assertions to the reification ex:claim
g.add((EX.claim, EX.source, EX.HRSystem))

tt1 = (EX.alice, EX.worksFor, EX.AcmeCorp)
tt2 = (EX.alice, EX.likes, EX.ProductABC)

# reifiers(): returns the reifier node(s) that reify a given triple term.
print([g.qname(r) for r in g.reifiers(TT=tt1)])

# reifications(): returns a list of triple terms that have at least one reifier.
for tt in g.reifications():
    print(tt)

# reifier_annotations(): returns a reifier's annotation triples (excludes rdf:reifies itself)
for reifier, pred, val in g.reifier_annotations(tt1):
    print(g.qname(reifier), g.qname(pred), g.qname(val))

# reified_triples(): returns the triple term(s) a specific reifier reifies
for tt in g.reified_triples(EX.claim):
    print(tt)

# give ex:claim a second rdf:reifies link, so scoped vs. wildcard removal are distinguishable
g.add((EX.claim, RDF.reifies, tt2))

# remove_reification(reifier, triple_term): remove the reification link between a reifier
# and the specified triple term only.
g.remove_reification(EX.claim, tt1)
print((EX.claim, RDF.reifies, tt1) in g)         # False - only this link removed
print((EX.claim, RDF.reifies, tt2) in g)         # True  - untouched
print((EX.claim, EX.source, EX.HRSystem) in g)  # True  - untouched

# remove_reification(reifier): removes all rdf:reifies links from the reifier
g.remove_reification(EX.claim)
print((EX.claim, RDF.reifies, tt2) in g)
print((EX.claim, EX.source, EX.HRSystem) in g)

print("\nResulting graph")
print(g.serialize(format="turtle12"))

['ex:claim']
<<( ex:alice ex:worksFor ex:AcmeCorp )>>
<<( ex:alice ex:likes ex:ProductABC )>>
ex:claim ex:source ex:HRSystem
<<( ex:alice ex:worksFor ex:AcmeCorp )>>
False
True
True
False
True

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:worksFor ex:GlobalTech .

ex:claim ex:source ex:HRSystem .

ex:other rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### Direction-tagged string literals

StarLayerGraph provides support for the directiona  language strngs introduced in RDF 1.2 through `DirLangString`. 

In [6]:
g = StarLayerGraph()
g.bind("ex", EX)

# literals can now include language direction
g.add((EX.ProductABC, EX.description, DirLangString("مرحبا", "ar", "rtl")))
g.add((EX.ProductABC, EX.description, Literal("hello", "en")))
g.add((EX.ProductABC, EX.description, DirLangString("hello", "en", "ltr")))

print(g.serialize(format="turtle12"))

@version "1.2" .
@prefix ex: <http://example.org/> .

ex:ProductABC ex:description "مرحبا"@ar--rtl, "hello"@en--ltr, "hello"@en .



## 2. Turtle 1.2 syntax

StarLayerGraph extends the ability of rdflib to parse Turtle 1.2, including triple terms and directed language strings.  

This section shows uses of  Turtle 1.2 syntax.  Output is made using longturtle12 sytnax to make it easier to understand the output graph.

See the [serialization formats guide](05a-serialization-formats.ipynb) for other RDF 1.2 serialization formats.

### 2.1 Canonical form: `rdf:reifies <<( s p o )>>`

The cannonical form — a triple term in `<<( )>>` syntax, reified by name via `rdf:reifies.`

In [7]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem .
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))

# note that the triple "ex:alice ex:worksFor ex:AcmeCorp" is never asserted into the graph.


triples: 2

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:claim ex:source ex:HRSystem .
ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



### 2.2 Reifying shorthand using: `<< s p o >>`

`<<( s p o )>>` above represents the triple *term* itself.  

`<< s p o >>` (no parens) is different: it is shorthand for creating an annonybous reifier with the provided predicate and object.  Similar to above, it does not assert the triple being reified. 

In [8]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))


triples: 2

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

_:rr_0 ex:source ex:HRSystem .
_:rr_0 rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



### 2.2a Using a named reifier

Adding an IRI preceded by `~` to the shorthand, such as `<< s p o ~ ex:id >>`, allows the addition of a named reifier to the triple term. This allows for the adding of additional triples to the reifier, which is not possible with an anonymous reifier.

In [9]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:id >> ex:source ex:HRSystem;  ex:reportedDate "2026-01-01"^^xsd:date .
   
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))

triples: 3

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:id ex:reportedDate "2026-01-01"^^xsd:date .
ex:id ex:source ex:HRSystem .
ex:id rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



### 2.2b Nesting a reifiedTriple

A `reifiedTriple` can be nested inside another — the object of one `<< >>` can itself be another `<< >>`, reifying a claim about a claim.

In [10]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    << ex:claim ex:about << ex:alice ex:worksFor ex:AcmeCorp >> ~ ex:outer >> ex:recordedBy ex:AuditSystem .
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))

triples: 3

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:outer ex:recordedBy ex:AuditSystem .
ex:outer rdf:reifies <<( ex:claim ex:about _:rr_0 )>> .
_:rr_0 rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .



### 2.2c Grouping references to triple term reifiers.

A comma-separated list of `<< s p o >>` values lets one subject/predicate point at several *distinct* triple terms at once,creating a way to group triple terms together through their reifiers.  

In [11]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:report123 ex:contains << ex:sensor1 ex:measured "20.1" ~ ex:r1 >> ,
                              << ex:sensor2 ex:measured "22.3" ~ ex:r2 >> .
    ex:r1 ex:source ex:SystemA .
    ex:r2 ex:source ex:SystemB .
""", format="turtle12")



print("\nThe triples ex:report123 contains:")
for s, p, reifier in g.triples((EX.report123, None, None)):
    for tt in g.reified_triples(reifier):
        print(" ", tt)

# assert the triples in the report
for s, p, reifier in g.triples((EX.report123, None, None)):
    for tt in g.reified_triples(reifier):
        g.add(tt)

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))


The triples ex:report123 contains:
  <<( ex:sensor1 ex:measured "20.1" )>>
  <<( ex:sensor2 ex:measured "22.3" )>>
triples: 8

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:r1 ex:source ex:SystemA .
ex:r1 rdf:reifies <<( ex:sensor1 ex:measured "20.1" )>> .
ex:r2 ex:source ex:SystemB .
ex:r2 rdf:reifies <<( ex:sensor2 ex:measured "22.3" )>> .
ex:report123 ex:contains ex:r1 .
ex:report123 ex:contains ex:r2 .
ex:sensor1 ex:measured "20.1" .
ex:sensor2 ex:measured "22.3" .



### 2.3 Inline annotation shorthand: `{| pred val |}`

The `{| ... |}` shorthand creates annotation triples at the same time that a triple is created.   

This example creates an annonymous reifier and attaches multiple annotation to that reifier.  

In [12]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))


# note: ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} asserts and reifies
# the triple at the same time.  ex:alice ex:likes ex:ProductABC appears in the graph.

triples: 4

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:likes ex:ProductABC .
_:rr_0 ex:since "2020" .
_:rr_0 ex:source ex:CRM .
_:rr_0 rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### 2.3a Using a named reifier. 

The same example as above using `~` to create a named reifier. 


In [13]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:likes ex:ProductABC ~ ex:id {| ex:since "2020" ; ex:source ex:CRM |} .
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))


# ~ ex:id is created as a named entity as the reifier.

triples: 4

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:likes ex:ProductABC .
ex:id ex:since "2020" .
ex:id ex:source ex:CRM .
ex:id rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### 2.3b Multiple annotation blocks. 

The same example as above expanded to use multiple annotation blocks.


In [14]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:likes ex:ProductABC  {| ex:since "2020" ; ex:source ex:CRM |} ~ ex:id2 {| ex:since "2019" ; ex:source ex:alice |}.
""", format="turtle12")

print("triples:", len(g))
print("\nResulting graph")
print(g.serialize(format="longturtle12"))


# Each block/reifier here produces its own independent reification of the *same*
# underlying triple (ex:alice ex:likes ex:ProductABC) - two separate claims about one fact,
# one via an anonymous reifier and one via the named ex:id2.

triples: 7

Resulting graph
@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .

ex:alice ex:likes ex:ProductABC .
ex:id2 ex:since "2019" .
ex:id2 ex:source ex:alice .
ex:id2 rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .
_:rr_0 ex:since "2020" .
_:rr_0 ex:source ex:CRM .
_:rr_0 rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .



### 2.5 Direction-tagged literals: `"text"@lang--dir`

The same `DirLangString` values built with Python above are written directly in Turtle 1.2 as an ordinary language-tagged literal with a `--ltr`/`--rtl` direction suffix.

In [15]:
g = StarLayerGraph()
g.bind("ex", EX)
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:listing_en ex:description "hello"@en--ltr .
    ex:listing_ar ex:description "مرحبا"@ar--rtl .
""", format="turtle12")

print("triples:", len(g))
print(g.serialize(format="turtle12"))

triples: 2
@version "1.2" .
@prefix ex: <http://example.org/> .

ex:listing_ar ex:description "مرحبا"@ar--rtl .

ex:listing_en ex:description "hello"@en--ltr .



## Further Reading on Graphs

1. **[Getting Started](01-getting-started.ipynb)** — install, first parse, first query, first validate.
2. **Graphs** — this guide.
   - 2.a **[Working with datasets](02a-graphs-datasets.ipynb)** — `StarLayerDataset`, multiple named graphs in one store.
   - 2.b **[Inferencing](02b-graphs-inferencing.ipynb)** — RDFS and OWL 2 RL reasoning via `owlrl`.
3. **[SPARQL](03-sparql.ipynb)** — query semantics and built-in functions.
5. **Other**
   - 5.a **[Serialization formats](05a-serialization-formats.ipynb)** — all supported RDF 1.2 formats.
   - 5.d **[Canonical hashing and graph comparison](05d-canonical-hashing.ipynb)** — RDFC-1.0 canonicalization/hashing and graph isomorphism.
